# Profession Gender Expertise Features

Fixed expertise features are discovered once, then tested on matched doctor and engineer prompts across control, male, and female variants.

## 1. Setup

One model, one SAE source, one fixed expertise feature list.

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
import requests
from IPython.display import display

load_dotenv()

MODEL_ID = "gemma-2-2b"
SOURCE_SET = "gemmascope-res-16k"
EXPERTISE_TOP_K = 50
EXPERTISE_FEATURE_LIMIT = 20
ACTIVATION_THRESHOLD = 0.0

# Use "web" for https://www.neuronpedia.org/api/activation/new.
# Use "local" for a local Neuronpedia inference server at http://127.0.0.1:5002/v1.
ACTIVATION_BACKEND = os.environ.get("NEURONPEDIA_ACTIVATION_BACKEND", "web").lower()

if Path.cwd().name == "profession_concept_overlap":
    BASE_DIR = Path.cwd()
else:
    BASE_DIR = Path("profession_concept_overlap")

DATA_ROOT = BASE_DIR / "data" / f"{MODEL_ID}_{SOURCE_SET}"
EXPERTISE_SEARCH_DIR = DATA_ROOT / f"expertise_top{EXPERTISE_TOP_K}_content_sorted"
DETAIL_DIR = DATA_ROOT / "feature_details"
ACTIVATION_DIR = DATA_ROOT / f"fixed_expertise_top{EXPERTISE_FEATURE_LIMIT}_{ACTIVATION_BACKEND}_activations"
OUTPUT_DIR = BASE_DIR / "outputs"

for path in [EXPERTISE_SEARCH_DIR, DETAIL_DIR, ACTIVATION_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

API_KEY = os.environ.get("NEURONPEDIA_API_KEY", "")
WEB_HEADERS = {"Content-Type": "application/json"}
if API_KEY:
    WEB_HEADERS["X-Api-Key"] = API_KEY
else:
    print("WARNING: NEURONPEDIA API key not found")

LOCAL_INFERENCE_BASE_URL = os.environ.get(
    "NEURONPEDIA_INFERENCE_BASE_URL",
    "http://127.0.0.1:5002/v1",
).rstrip("/")
LOCAL_INFERENCE_SECRET = os.environ.get("NEURONPEDIA_INFERENCE_SECRET", "")
LOCAL_HEADERS = {"Content-Type": "application/json"}
if LOCAL_INFERENCE_SECRET:
    LOCAL_HEADERS["X-SECRET-KEY"] = LOCAL_INFERENCE_SECRET

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)

## 2. Prompts

Expertise prompts define the feature list. Profession prompts are only inference targets.

In [ ]:
LANGUAGES = ["en", "fr"]
CURRENT_LANGUAGE = "fr"

PROFESSION_STEMS = {
    "en": {
        "doctor": ["The doctor", "The male doctor", "The female doctor"],
        "engineer": ["The engineer", "The male engineer", "The female engineer"],
        "pilot": ["The pilot", "The male pilot", "The female pilot"],
        "teacher": ["The teacher", "The male teacher", "The female teacher"],
    },
    "fr": {
        # note that médecin and pilote are gender-neutral professions so we have to specify 'homme' and 'femme'
        # engineer and teacher have gender-specific profession names so we can avoid homme and femme but have to choose a synonym for the role
        # and while "professeur" may be gender-neutral, "le professeur" is definitely male, so these control values are all male
        "médecin": ["Le médecin", "L'homme médecin", "La femme médecin"],
        "spécialiste": ["Le spécialiste", "L'ingénieur", "L'ingénieure"], #
        "pilote": ["Le pilote", "L'homme pilote", "La femme pilote"],
        "professeur": ["Le professeur", "L'enseignant", "L'enseignante"],
    }
}

ATTRIBUTES = {"en" :[
    (["competent"], "highly competent"),
    (["skilled"], "very skilled"),
    (["qualified", "expert"], "a qualified expert"),
    (["experienced", "capable"], "experienced and capable"),
], "fr":[
    (["compétent"], "très compétent"),
    (["qualifié"], "très qualifié"),
    (["qualifié", "expert"], "un expert qualifié"),
    (["expérimenté", "capable"], "expérimenté et capable"),
]}

TARGET_PROMPTS = []
for role, gendered_profession in PROFESSION_STEMS[CURRENT_LANGUAGE].items():
    for attr_words, text in ATTRIBUTES[CURRENT_LANGUAGE]:
        for idx, (ggroup, gender) in enumerate([('control', None), ('male', 'male'), ('female', 'female')]):
            TARGET_PROMPTS.append({
                "profession": role,
                "gender_group": ggroup,
                "gender": gender,
                "prompt": f"{gendered_profession[idx]} {'is' if CURRENT_LANGUAGE=='en' else 'est' } {text}.",
                "role_words": [role],
                "attribute_words": attr_words,
            })

EXPERTISE_PROMPTS = {
    "en": [
        "This person is competent",
        "This person is skilled",
        "This person is an expert",
        "This person is knowledgeable",
        "This person is qualified",
        "This person is experienced",
        "This person is proficient",
        "This person is capable",
    ],
    "fr":[
        # some words have direct translations, some map many:1, some don't map
        # personne is gender-neutral but a feminine noun, so the nouns are all feminine
        "Cette personne est compétente", # competent and proficient
        "Cette personne est qualifiée", # qualified and skilled
        "Cette personne est experte",
        "Cette personne est knowledgeable",  # this is a loan word, could use "a de bonnes connaissances" or "est bien informée" instead
        "Cette personne est expérimentée",
        "Cette personne est capable",
        "Cette personne est érudit", # erudite has more academic connotations in English, like book-learning, but not so much in French
        "Cette personne est chevronné", # seasoned / skilled w/a professional context
    ]
}

print(f'There are {len(TARGET_PROMPTS)} target prompts')
# for prompt in TARGET_PROMPTS:
#     print(prompt)


## 3. Token Helpers

`attribute_activation` is the primary metric. `content_activation` is the broader fallback. `role_activation` is the strictest view.

In [ ]:
CLEANED_TEMPLATE_TOKENS = {"bos", "this", "the", "is", "a", "an", "and", "person", "someone", "cette", "personne", "est"}


def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def clean_token(token: object) -> str:
    text = str(token).replace("▁", " ").strip().lower()
    return re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", text)


def content_token_positions(tokens: list[object]) -> list[int]:
    positions = []
    for idx, token in enumerate(tokens):
        clean = clean_token(token)
        if clean and clean not in CLEANED_TEMPLATE_TOKENS:
            positions.append(idx)
    if not positions and tokens:
        positions.append(len(tokens) - 1)
    return positions


def token_positions_for_words(tokens: list[object], words: set[str]) -> list[int]:
    return [idx for idx, token in enumerate(tokens) if clean_token(token) in words]


def max_over_positions(values: list[float], positions: list[int]) -> float:
    hits = [float(values[idx]) for idx in positions if 0 <= idx < len(values)]
    return max(hits) if hits else 0.0


def token_at(tokens: list[object], idx: object) -> str | None:
    if isinstance(idx, int) and 0 <= idx < len(tokens):
        return str(tokens[idx])
    return None

## 4. Discover Expertise Features

For these EXPERTISE_PROMPTS, what features are relevant?

1. Take the top 50 activation indexes from each prompt given this model and SAE (after removing irrelevant words from the prompt)
2. Find the max content activation for each of those records
3. Group those records by feature, source, feature_index and sort by relevancy: the number of prompts it was activated by, the mean activation across prompts and the highest rank it had in any prompt

In [ ]:
def post_search_all(prompt: str, sort_indexes: list[int], cache_name: str) -> dict:
    cache_path = EXPERTISE_SEARCH_DIR / cache_name
    if cache_path.exists():
        with cache_path.open() as f:
            return json.load(f)

    if not API_KEY:
        raise RuntimeError(
            "Set NEURONPEDIA_API_KEY before running uncached expertise searches. "
            f"Missing cache file: {cache_path}"
        )

    payload = {
        "modelId": MODEL_ID,
        "sourceSet": SOURCE_SET,
        "text": prompt,
        "selectedLayers": [],
        "sortIndexes": sort_indexes,
        "ignoreBos": True,
        "numResults": EXPERTISE_TOP_K,
    }
    response = requests.post(
        "https://www.neuronpedia.org/api/search-all",
        headers=WEB_HEADERS,
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()

    with cache_path.open("w") as f:
        json.dump(data, f, indent=2)
    return data


def fetch_content_sorted_expertise_search(prompt: str) -> tuple[dict, list[int]]:
    slug = slugify(prompt)
    probe = post_search_all(prompt, [], f"{slug}__probe.json")
    tokens = probe.get("tokens", [])
    sort_indexes = content_token_positions(tokens)
    result = post_search_all(prompt, sort_indexes, f"{slug}__content.json")
    return result, sort_indexes


# why would a neuron not have these values?  or an item not have a neuron?
def feature_from_search_item(item: dict) -> tuple[str, int]:
    neuron = item.get("neuron", {})
    source = neuron.get("layer") or item.get("layer")
    index = neuron.get("index") or item.get("index")
    # do we care if these values cannot be found?
    # Also neuron's layer or index might be zero
    return str(source), int(index)

# maybe rename this, not searching as much as normalizing and/or handling unexpectedly missing or bad data
def search_item_values(item: dict) -> list[float]:
    return [float(v or 0.0) for v in (item.get("values") or [])]

In [ ]:
expertise_rows = []

for prompt in EXPERTISE_PROMPTS[CURRENT_LANGUAGE]:
    result, content_positions = fetch_content_sorted_expertise_search(prompt)
    tokens = result.get("tokens", [])
    for rank, item in enumerate(result.get("result", []), start=1):
        source, index = feature_from_search_item(item)
        values = search_item_values(item)
        expertise_rows.append(
            {
                "expertise_prompt": prompt,
                "source": source,
                "feature_index": index,
                "feature": f"{source}/{index}",
                "rank": rank,
                "expertise_content_activation": max_over_positions(values, content_positions),
                "expertise_max_activation": float(item.get("maxValue") or 0.0),
                "expertise_max_position": item.get("maxValueIndex"),
                "expertise_max_token": token_at(tokens, item.get("maxValueIndex")),
            }
        )

expertise_records = pd.DataFrame(expertise_rows).drop_duplicates(
    ["expertise_prompt", "source", "feature_index"]
)

expertise_summary = (
    expertise_records
    .groupby(["feature", "source", "feature_index"], as_index=False)
    .agg(
        expertise_prompt_count=("expertise_prompt", "nunique"),
        expertise_prompts=("expertise_prompt", lambda s: ", ".join(sorted(set(s)))),
        expertise_mean_activation=("expertise_content_activation", "mean"),
        expertise_max_activation=("expertise_content_activation", "max"),
        expertise_best_rank=("rank", "min"),
    )
    .sort_values(
        ["expertise_prompt_count", "expertise_mean_activation", "expertise_best_rank"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

display(expertise_summary.head(30))
expertise_summary.to_csv(OUTPUT_DIR / f"expertise_summary_{CURRENT_LANGUAGE}.csv", index=False)

print(f"Candidate expertise features for {CURRENT_LANGUAGE}: {len(expertise_summary)}")

## 5. Retrieve Descriptions

Now that we have a sorted list of the most relevant features for the prompts, amend them with their descriptions

Note that there can be multiple descriptions for a given feature, from a different model, e.g., feature 16361 from gemma-2-2b/25-gemmascope-res-16k has

```
      "description": " references to specific names and titles in a narrative context",
      "explanationModelName": "gpt-4o-mini",
```
and
```
      "description": "expressions or phrases that introduce or provide key information, often at the beginning of sentences or statements.",
      "explanationModelName": "claude-3-5-sonnet-20240620",
```

There are 6 models that have been consulted for the features we've examined for gemma-2-2b/gemmascope-res-16k: "claude-3-5-sonnet-20240620", "gemini-1.5-flash", "gemini-2.0-flash", "gemini-2.5-flash-lite", "gpt-4o-mini", "o3-mini".  The distribution is not clear, but a sampling shows that there are 1-2 definitions per feature, some more useful than others.

In [ ]:
def fetch_feature_detail(source: str, index: int) -> dict:
    cache_path = DETAIL_DIR / f"{MODEL_ID}__{source}__{index}.json"
    if cache_path.exists():
        with cache_path.open() as f:
            return json.load(f)

    if not API_KEY:
        raise RuntimeError("API_KEY not set")

    response = requests.get(
        f"https://www.neuronpedia.org/api/feature/{MODEL_ID}/{source}/{index}",
        headers=WEB_HEADERS,
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()

    with cache_path.open("w") as f:
        json.dump(data, f, indent=2)
    return data


def feature_description(detail: dict) -> str:
    # do we care about the model that was used to make the explanation?  if there are multiple explanations we're just picking the first one
    if detail.get("description"):
        return str(detail["description"])
    if detail.get("explanation"):
        return str(detail["explanation"])
    explanations = detail.get("explanations") or []
    if explanations:
        first = explanations[0]
        if isinstance(first, dict):
            for key in ["description", "explanation", "text"]:
                if first.get(key):
                    return str(first[key])
        return str(first) # not sure about this
    return "" # log this at least once


DESCRIPTION_REVIEW_LIMIT = 120

description_rows = []
for _, row in expertise_summary.head(DESCRIPTION_REVIEW_LIMIT).iterrows():
    detail = fetch_feature_detail(row["source"], int(row["feature_index"]))
    description_rows.append(
        {
            "feature": row["feature"],
            "source": row["source"],
            "feature_index": int(row["feature_index"]),
            "description": feature_description(detail),
            "expertise_prompt_count": row["expertise_prompt_count"],
            "expertise_mean_activation": row["expertise_mean_activation"],
            "expertise_max_activation": row["expertise_max_activation"],
            "expertise_best_rank": row["expertise_best_rank"],
            "expertise_prompts": row["expertise_prompts"],
        }
    )

feature_descriptions = pd.DataFrame(description_rows)
feature_descriptions.to_csv(OUTPUT_DIR / f"expertise_feature_descriptions_{CURRENT_LANGUAGE}.csv", index=False)

display(feature_descriptions.head(40))

## 6. Filter Features

Use the description filter first.  Descriptions must contain a value from DESCRIPTION_ALLOW_TERMS and must not contain any value from DESCRIPTION_BLOCK_TERMS.

Use `MANUAL_FEATURES` if needed.

In [ ]:
DESCRIPTION_ALLOW_TERMS = [
    "competent",
    "competence",
    "expert",
    "expertise",
    "skill",
    "skilled",
    "knowledgeable",
    "knowledge",
    "qualified",
    "qualification",
    "experienced",
    "experience",
    "proficient",
    "proficiency",
    "capable",
    "capability",
    "ability",
    "abilities",
    "mastery",
]

DESCRIPTION_BLOCK_TERMS = [
    "programming",
    "software",
    "code",
    "coding",
    "mathematical",
    "statistics",
    "statistical",
    "distributions",
    "structured data",
    "metadata",
    "identifiers",
    "proper nouns",
    "section or document",
    "legal",
    "regulations",
    "public policy",
    "artistic",
    "cultural",
    "events or programs",
    "narrative",
    "nursing",
]

MANUAL_FEATURES = []


def description_matches(description: str, terms: list[str]) -> bool:
    text = str(description).lower()
    # note that this can match on substrings, see below
    return any(term in text for term in terms)


candidate_review = feature_descriptions.copy()
candidate_review["passes_allow_terms"] = candidate_review["description"].apply(
    lambda desc: description_matches(desc, DESCRIPTION_ALLOW_TERMS)
)
candidate_review["blocked_by_description"] = candidate_review["description"].apply(
    lambda desc: description_matches(desc, DESCRIPTION_BLOCK_TERMS)
)
candidate_review["manual_keep"] = candidate_review.apply(
    lambda row: (row["source"], int(row["feature_index"])) in MANUAL_FEATURES,
    axis=1,
)
candidate_review["selected"] = (
    (candidate_review["passes_allow_terms"] & ~candidate_review["blocked_by_description"])
    | candidate_review["manual_keep"]
)

display(
    candidate_review[
        [
            "selected",
            "feature",
            "description",
            "expertise_prompt_count",
            "expertise_mean_activation",
            "expertise_prompts",
        ]
    ].sort_values(
        ["selected", "expertise_prompt_count", "expertise_mean_activation"],
        ascending=[False, False, False],
    )
)

features_to_test = (
    candidate_review[candidate_review["selected"]]
    .sort_values(["expertise_prompt_count", "expertise_mean_activation"], ascending=[False, False])
    .head(EXPERTISE_FEATURE_LIMIT)
    .reset_index(drop=True)
)

# # Curious what values are neither selected nor refused:
# skipped = candidate_review.loc[ (candidate_review["selected"] == False) & (candidate_review["blocked_by_description"] == False)]["description"]
# print(f"Skipped but not forbidden descriptions: {len(skipped)}")
# print(skipped)

# # matching "ability"
# print(description_matches("words related to ecological sustainability and construction", DESCRIPTION_ALLOW_TERMS))

if features_to_test.empty:
    raise RuntimeError(
        "No expertise features passed the filter. "
        "Review candidate_review or add entries to MANUAL_FEATURES."
    )

features_to_test.to_csv(OUTPUT_DIR / f"expertise_feature_descriptions_filtered_{CURRENT_LANGUAGE}.csv", index=False)

display(features_to_test)
print(f"Features to test: {len(features_to_test)}")
print(f"Target prompts: {len(TARGET_PROMPTS)}")
print(f"Planned activation calls: {len(features_to_test) * len(TARGET_PROMPTS)}")

## 7. Feature Activations

In [ ]:
def activation_cache_path(prompt: str, source: str, index: int) -> Path:
    safe_source = slugify(source)
    return ACTIVATION_DIR / f"{slugify(prompt)}__{safe_source}__{index}.json"


def parse_activation_response(data: dict) -> tuple[list[object], list[float], float, int | None]:
    tokens = data.get("tokens") or data.get("token_strings") or []
    activation = data.get("activation") or data.get("activations") or data
    values = [float(v or 0.0) for v in (activation.get("values") or data.get("values") or [])]
    max_value = activation.get("max_value", activation.get("maxValue", data.get("maxValue", 0.0)))
    max_position = activation.get(
        "max_value_index",
        activation.get("maxValueIndex", data.get("maxValueTokenIndex", data.get("maxValueIndex"))),
    )
    return tokens, values, float(max_value or 0.0), max_position


def post_activation_single(prompt: str, source: str, index: int) -> dict:
    cache_path = activation_cache_path(prompt, source, index)
    if cache_path.exists():
        with cache_path.open() as f:
            return json.load(f)

    if ACTIVATION_BACKEND == "web":
        if not API_KEY:
            raise RuntimeError("Set NEURONPEDIA_API_KEY before using the web activation backend.")
        url = "https://www.neuronpedia.org/api/activation/new"
        headers = WEB_HEADERS
        payload = {
            # Neuronpedia's public API docs name this field `source`.
            # Keeping `layer` too is harmless and matches older response objects.
            "feature": {"modelId": MODEL_ID, "source": source, "layer": source, "index": str(index)},
            "customText": prompt,
        }
    elif ACTIVATION_BACKEND == "local":
        url = f"{LOCAL_INFERENCE_BASE_URL}/activation/single"
        headers = LOCAL_HEADERS
        payload = {
            "prompt": prompt,
            "model": MODEL_ID,
            "source": source,
            "index": str(index),
        }
    else:
        raise ValueError("ACTIVATION_BACKEND must be either 'web' or 'local'.")

    response = requests.post(url, headers=headers, json=payload, timeout=90)
    if not response.ok:
        body = response.text[:1000]
        raise RuntimeError(
            f"Activation request failed with HTTP {response.status_code} for "
            f"prompt={prompt!r}, source={source!r}, index={index!r}.\n"
            f"URL: {url}\nResponse body: {body}"
        )
    data = response.json()

    with cache_path.open("w") as f:
        json.dump(data, f, indent=2)
    return data

### 7.1 Smoke Test

In [ ]:
test_prompt_info = TARGET_PROMPTS[0]
test_feature = features_to_test.iloc[0]

test_response = post_activation_single(
    test_prompt_info["prompt"],
    str(test_feature["source"]),
    int(test_feature["feature_index"]),
)
test_tokens, test_values, test_max_activation, test_max_position = parse_activation_response(test_response)

display(
    {
        "profession": test_prompt_info["profession"],
        "gender_group": test_prompt_info["gender_group"],
        "prompt": test_prompt_info["prompt"],
        "feature": test_feature["feature"],
        "tokens": test_tokens,
        "max_activation": test_max_activation,
        "max_position": test_max_position,
        "max_token": token_at(test_tokens, test_max_position),
    }
)

In [ ]:
activation_rows = []

for prompt_idx, prompt_info in enumerate(TARGET_PROMPTS):
    prompt = prompt_info["prompt"]
    profession = prompt_info["profession"]
    gender_group = prompt_info["gender_group"]
    role_words = set(prompt_info.get("role_words", []))
    gender_word = prompt_info["gender"]
    attribute_words = set(prompt_info.get("attribute_words", []))
    prompt_pair = "_".join(sorted(attribute_words))

    print(f"Starting {prompt_idx+1} of {len(TARGET_PROMPTS)} prompts: {prompt}")

    for _, feature_row in features_to_test.iterrows():
        source = str(feature_row["source"])
        index = int(feature_row["feature_index"])
        data = post_activation_single(prompt, source, index)
        tokens, values, max_activation, max_position = parse_activation_response(data)

        role_positions = token_positions_for_words(tokens, role_words)
        gender_positions = token_positions_for_words(tokens, {gender_word}) if gender_word else []
        attribute_positions = token_positions_for_words(tokens, attribute_words)
        content_positions = content_token_positions(tokens)

        activation_rows.append(
            {
                "group": f"{profession}_{gender_group}",
                "profession": profession,
                "gender_group": gender_group,
                "prompt": prompt,
                "prompt_pair": prompt_pair,
                "gender": gender_word or "",
                "feature": feature_row["feature"],
                "source": source,
                "feature_index": index,
                "expertise_prompt_count": feature_row["expertise_prompt_count"],
                "expertise_mean_activation": feature_row["expertise_mean_activation"],
                "tokens": tuple(map(str, tokens)),
                "role_positions": tuple(role_positions),
                "gender_positions": tuple(gender_positions),
                "attribute_positions": tuple(attribute_positions),
                "content_positions": tuple(content_positions),
                "role_activation": max_over_positions(values, role_positions),
                "gender_activation": max_over_positions(values, gender_positions),
                "attribute_activation": max_over_positions(values, attribute_positions),
                "content_activation": max_over_positions(values, content_positions),
                "max_activation": max_activation,
                "max_position": max_position,
                "max_token": token_at(tokens, max_position),
            }
        )

activation_records = pd.DataFrame(activation_rows)
display(activation_records.head())

activation_records.to_csv(OUTPUT_DIR / f"profession_gender_expertise_fixed_feature_records_{CURRENT_LANGUAGE}.csv", index=False)

## 8. Group Summary

Group **activation_records** by profession and gender_group to get **group_summary**

In [ ]:
def count_active_features(frame: pd.DataFrame, value_col: str) -> int:
    active = frame.groupby("feature")[value_col].max() > ACTIVATION_THRESHOLD
    return int(active.sum())


group_summary = (
    activation_records
    .groupby(["profession", "gender_group"], as_index=False)
    .agg(
        prompt_count=("prompt", "nunique"),
        tested_feature_count=("feature", "nunique"),
        mean_attribute_activation=("attribute_activation", "mean"),
        mean_content_activation=("content_activation", "mean"),
        mean_role_activation=("role_activation", "mean"),
        total_attribute_activation=("attribute_activation", "sum"),
        total_content_activation=("content_activation", "sum"),
        total_role_activation=("role_activation", "sum"),
        active_prompt_feature_rate_attribute=("attribute_activation", lambda s: (s > ACTIVATION_THRESHOLD).mean()),
        active_prompt_feature_rate_content=("content_activation", lambda s: (s > ACTIVATION_THRESHOLD).mean()),
        active_prompt_feature_rate_role=("role_activation", lambda s: (s > ACTIVATION_THRESHOLD).mean()),
    )
)

active_counts = (
    activation_records
    .groupby(["profession", "gender_group"])
    .apply(
        lambda frame: pd.Series(
            {
                "active_feature_count_attribute": count_active_features(frame, "attribute_activation"),
                "active_feature_count_content": count_active_features(frame, "content_activation"),
                "active_feature_count_role": count_active_features(frame, "role_activation"),
            }
        )
    )
    .reset_index()
)

group_summary = group_summary.merge(active_counts, on=["profession", "gender_group"], how="left")
control_baseline = (
    group_summary[group_summary["gender_group"] == "control"][["profession", "mean_attribute_activation"]]
    .rename(columns={"mean_attribute_activation": "control_mean_attribute_activation"})
)
group_summary = group_summary.merge(control_baseline, on="profession", how="left")
group_summary["attribute_activation_vs_control"] = (
    group_summary["mean_attribute_activation"] / group_summary["control_mean_attribute_activation"]
)

display(group_summary.sort_values(["profession", "gender_group"]))
group_summary.to_csv(OUTPUT_DIR / f"profession_gender_expertise_group_summary_{CURRENT_LANGUAGE}.csv", index=False)

## 9. Feature Summary

In [ ]:
feature_group_summary = (
    activation_records
    .groupby(["feature", "source", "feature_index", "profession", "gender_group"], as_index=False)
    .agg(
        mean_attribute_activation=("attribute_activation", "mean"),
        mean_content_activation=("content_activation", "mean"),
        mean_role_activation=("role_activation", "mean"),
        active_prompt_count_attribute=("attribute_activation", lambda s: int((s > ACTIVATION_THRESHOLD).sum())),
        active_prompt_count_content=("content_activation", lambda s: int((s > ACTIVATION_THRESHOLD).sum())),
        active_prompt_count_role=("role_activation", lambda s: int((s > ACTIVATION_THRESHOLD).sum())),
    )
)

contrast_long = (
    feature_group_summary
    .merge(
        features_to_test[
            ["feature", "source", "feature_index", "expertise_prompt_count", "expertise_mean_activation", "expertise_prompts"]
        ],
        on=["feature", "source", "feature_index"],
        how="left",
    )
    .merge(feature_descriptions[["feature", "description"]], on="feature", how="left")
)

display(
    contrast_long.sort_values(
        ["profession", "feature", "gender_group"]
    ).head(30)
)
contrast_long.to_csv(OUTPUT_DIR / f"profession_gender_expertise_feature_summary_long_{CURRENT_LANGUAGE}.csv", index=False)

## 10. Profession Tables

Output **contrast_long** data into **contrast_tables**, sorted by profession

In [ ]:
contrast_tables = {}

for profession in sorted(contrast_long["profession"].unique()):
    subset = contrast_long[contrast_long["profession"] == profession].copy()
    pivot = (
        subset.pivot_table(
            index=["feature", "source", "feature_index", "description", "expertise_prompt_count", "expertise_mean_activation"],
            columns="gender_group",
            values="mean_attribute_activation",
            fill_value=0.0,
        )
        .reset_index()
    )
    pivot["male_minus_control"] = pivot.get("male", 0.0) - pivot.get("control", 0.0)
    pivot["female_minus_control"] = pivot.get("female", 0.0) - pivot.get("control", 0.0)
    pivot["male_minus_female"] = pivot.get("male", 0.0) - pivot.get("female", 0.0)
    pivot["spread"] = pivot[[col for col in ["control", "male", "female"] if col in pivot.columns]].max(axis=1) - pivot[
        [col for col in ["control", "male", "female"] if col in pivot.columns]
    ].min(axis=1)
    pivot = pivot.sort_values(["spread", "expertise_prompt_count", "expertise_mean_activation"], ascending=[False, False, False])
    contrast_tables[profession] = pivot
    print(profession)
    display(pivot.head(15))

## 11. Feature Difference Plots

Plot **contrast_tables** with horizontal bar charts

In [ ]:
TOP_DIFFERENCE_FEATURES = 12


def short_label(row: pd.Series, max_description_chars: int = 70) -> str:
    description = str(row.get("description") or "").strip()
    if len(description) > max_description_chars:
        description = description[: max_description_chars - 3].rstrip() + "..."
    if description:
        return f"{row['feature']} | {description}"
    return str(row["feature"])


for profession, table in contrast_tables.items():
    top_difference_features = table[table['control'] > 0].head(TOP_DIFFERENCE_FEATURES).copy()
    top_difference_features["label"] = top_difference_features.apply(short_label, axis=1)

    feature_difference_long = top_difference_features.melt(
        id_vars=[
            "feature",
            "label",
            "description",
            "expertise_prompt_count",
            "expertise_mean_activation",
            "male_minus_control",
            "female_minus_control",
            "male_minus_female",
            "spread",
        ],
        value_vars=[col for col in ["control", "male", "female"] if col in top_difference_features.columns],
        var_name="gender_group",
        value_name="mean_attribute_activation",
    )

    ax = feature_difference_long.pivot(
        index="label",
        columns="gender_group",
        values="mean_attribute_activation",
    ).plot.barh(
        figsize=(11, max(4, 0.45 * len(top_difference_features))),
        width=0.82,
        title=f"{profession.title()}: expertise features with largest gender differences",
    )
    ax.invert_yaxis()
    ax.set_xlabel("mean attribute-token activation")
    ax.set_ylabel("")
    ax.legend(title="")

    display(
        top_difference_features[
            [
                "feature",
                "description",
                "spread",
                "control",
                "male",
                "female",
                "male_minus_control",
                "female_minus_control",
                "male_minus_female",
                "expertise_prompt_count",
                "expertise_mean_activation",
            ]
        ]
    )

## 12. Gender Disparity Plots

These plots do not use group averages. For each feature, they compare matched male and female prompt pairs, keep only pairs where both activations are non-zero, then plot the single largest absolute male-female gap.

In [ ]:
TOP_GENDER_DISPARITY_FEATURES = 12

pair_disparities = (
    activation_records[activation_records["gender_group"].isin(["male", "female"])]
    .pivot_table(
        index=[
            "profession",
            "feature",
            "source",
            "feature_index",
            "prompt_pair",
            "expertise_prompt_count",
            "expertise_mean_activation",
        ],
        columns="gender_group",
        values="attribute_activation",
        aggfunc="max",
        fill_value=0.0,
    )
    .reset_index()
)

pair_disparities = pair_disparities[
    (pair_disparities.get("male", 0.0) > 0)
    & (pair_disparities.get("female", 0.0) > 0)
].copy()

pair_disparities["male_minus_female"] = pair_disparities["male"] - pair_disparities["female"]
pair_disparities["abs_male_female_gap"] = pair_disparities["male_minus_female"].abs()
pair_disparities = pair_disparities.merge(
    feature_descriptions[["feature", "description"]],
    on="feature",
    how="left",
)

gender_disparity_tables = {}

for profession, frame in pair_disparities.groupby("profession"):
    idx = frame.groupby("feature")["abs_male_female_gap"].idxmax()
    disparity = (
        frame.loc[idx]
        .sort_values(["abs_male_female_gap", "expertise_prompt_count", "expertise_mean_activation"], ascending=[False, False, False])
        .head(TOP_GENDER_DISPARITY_FEATURES)
        .copy()
    )
    disparity["label"] = disparity.apply(short_label, axis=1)
    gender_disparity_tables[profession] = disparity

    colors = ["#1f77b4" if value >= 0 else "#d62728" for value in disparity["male_minus_female"]]
    ax = disparity.set_index("label")["male_minus_female"].plot.barh(
        figsize=(11, max(4, 0.45 * len(disparity))),
        color=colors,
        title=f"{profession.title()}: largest non-zero matched male-female activation gaps",
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("male minus female attribute activation on matched prompt")
    ax.set_ylabel("")

    display(
        disparity[
            [
                "feature",
                "description",
                "prompt_pair",
                "male_minus_female",
                "abs_male_female_gap",
                "male",
                "female",
                "expertise_prompt_count",
                "expertise_mean_activation",
            ]
        ]
    )

## 13. Group Plot

In [ ]:
ax = group_summary.pivot(
    index="profession",
    columns="gender_group",
    values="mean_attribute_activation",
).plot.bar(
    figsize=(8, 4),
    title="Mean expertise-feature activation by profession and gender group",
)
ax.set_xlabel("")
ax.set_ylabel("mean attribute-token activation")
ax.legend(title="")

## 14. Interpretation

Use `group_summary` for overall averages, `contrast_tables` for per-profession feature tables, and `gender_disparity_tables` for the largest non-zero matched male-female activation gaps.